# 00 — Fixed train subsample and validation split

Every model in this homework is trained on the same `splits/train_ids.csv` and scored on the same
`splits/val_ids.csv`, so all comparisons use one fixed validation split.

1. Stratified on `domain × label` using only the label columns (texts are not loaded for this): 10k validation
   rows, then a 40k training subsample from the rest.
2. **Near-duplicate check.** A fusion text is a human text partly rewritten by an LLM, so the same source text
   can sit in train as `human` and in validation as `fusion`. Validation would then reward memorising the
   source. MinHash (word 5-grams) finds such pairs; matching training rows are dropped.
3. The id files are written once. Set `OVERWRITE = True` only if you deliberately want a new split.

**Memory:** only the 50k selected texts are read (streamed from `train.csv`), and MinHash signatures are built one
text at a time, so peak RAM stays around 1 GB. Restart other kernels that still hold the full `train.csv`.

In [1]:
import sys
from pathlib import Path

# make src/ importable whether the kernel starts in src/experiments, src or the repo root
SRC = next(q for p in [Path.cwd(), *Path.cwd().parents] for q in (p, p / "src") if (q / "config.py").exists())
sys.path.insert(0, str(SRC))

In [2]:
import gc
import time

import pandas as pd
from datasketch import LeanMinHash, MinHash, MinHashLSH
from sklearn.model_selection import train_test_split

from config import LABELS, SEED, SPLITS_DIR, TRAIN_SIZE, VAL_SIZE
from utils import load_meta, load_rows

OVERWRITE = False

SHINGLE = 5            # word n-gram size for near-duplicate detection
NUM_PERM = 128         # MinHash permutations (more = more precise Jaccard estimate)
MAX_WORDS = 2_000      # shingles from the first 2k words only (90% of texts are shorter)
LSH_THRESHOLD = 0.2    # recall-oriented candidate search; exact Jaccard is computed afterwards
DROP_THRESHOLD = 0.2   # drop a training row if it shares >= 20% of 5-gram shingles with a validation row

In [3]:
meta = load_meta()                                   # id, lang, domain, label - no texts
strata = meta["domain"] + "|" + meta["label"]

rest, val_meta = train_test_split(meta, test_size=VAL_SIZE, stratify=strata, random_state=SEED)
train_meta, _ = train_test_split(rest, train_size=TRAIN_SIZE, stratify=strata.loc[rest.index], random_state=SEED)
del meta, rest, strata; gc.collect()

rows = load_rows(pd.concat([train_meta["id"], val_meta["id"]]))
train, val = rows.iloc[:len(train_meta)].reset_index(drop=True), rows.iloc[len(train_meta):].reset_index(drop=True)
del rows; gc.collect()
print(f"val={len(val):,}  train candidates={len(train):,}")

val=10,000  train candidates=40,000


In [7]:
train.head()

,id,text,lang,domain,label
0,75d2f2,On the status of pointlike ﬁelds in integrable...,en-EN,arxiv_papers,fusion
1,9fe647,"Liebe Kolleginnen und Kollegen, verehrte Damen...",de-DE,bundestag,ai
2,95ecd7,"Man, I’ve gotta tell you about the past couple...",en-EN,blog_authorship_corpus,ai
3,df4220,So one day this girl met this guy. This girl t...,en-EN,blog_authorship_corpus,fusion
4,9fbd6a,By . Daily Mail Reporter . PUBLISHED: . 12:34 ...,en-EN,cnn_news,human


## Near-duplicate check between the training subsample and validation

In [4]:
def shingles(text: str) -> set:
    w = text.lower().split()[:MAX_WORDS]
    return {" ".join(w[i:i + SHINGLE]) for i in range(max(1, len(w) - SHINGLE + 1))}


_base = MinHash(num_perm=NUM_PERM, seed=SEED)

def signature(text: str) -> LeanMinHash:
    m = _base.copy()
    m.update_batch([s.encode("utf8") for s in shingles(text)])
    return LeanMinHash(m)                             # drops the permutation arrays -> small


t0 = time.time()
lsh = MinHashLSH(threshold=LSH_THRESHOLD, num_perm=NUM_PERM)
for i, text in enumerate(train["text"]):              # streamed: one text in memory at a time
    lsh.insert(i, signature(text))
print(f"indexed {len(train):,} training texts [{time.time() - t0:.0f}s]")

tr_text = train["text"].to_numpy()
pairs = []
for j, text in enumerate(val["text"]):
    cand = lsh.query(signature(text))
    if not cand:
        continue
    b = shingles(text)
    for i in cand:
        a = shingles(tr_text[i])
        jac = len(a & b) / len(a | b)
        if jac >= DROP_THRESHOLD:
            pairs.append((i, j, jac))
del lsh; gc.collect()
pairs = pd.DataFrame(pairs, columns=["tr_pos", "va_pos", "jaccard"])
print(f"near-duplicate pairs (Jaccard >= {DROP_THRESHOLD}): {len(pairs):,}   [{time.time() - t0:.0f}s]")

indexed 40,000 training texts [18s]
near-duplicate pairs (Jaccard >= 0.2): 1,452   [23s]


In [5]:
if len(pairs):
    pairs["train_id"] = train["id"].to_numpy()[pairs.tr_pos]
    pairs["val_id"] = val["id"].to_numpy()[pairs.va_pos]
    pairs["train_label"] = train["label"].to_numpy()[pairs.tr_pos]
    pairs["val_label"] = val["label"].to_numpy()[pairs.va_pos]
    pairs["domain"] = val["domain"].to_numpy()[pairs.va_pos]
    print("Label pairs (rows = training label, cols = validation label):")
    print(pd.crosstab(pairs.train_label, pairs.val_label).to_string())
    print("\nJaccard distribution:")
    print(pairs.jaccard.describe(percentiles=[.25, .5, .75]).round(3).to_string())
    print("\nPairs per domain:")
    print(pairs.domain.value_counts().to_string())

Label pairs (rows = training label, cols = validation label):
val_label     ai  fusion  human
train_label                    
ai           162       0      0
fusion         0     416    405
human          1     423     45

Jaccard distribution:
count    1452.000
mean        0.496
std         0.203
min         0.200
25%         0.327
50%         0.463
75%         0.630
max         1.000

Pairs per domain:
domain
student_essays            330
house_of_commons          245
bundestag                 238
blog_authorship_corpus    209
euro_court_cases          192
cnn_news                  122
arxiv_papers               91
gutenberg                  25


If many `human ↔ fusion` pairs appear, that confirms the leak and is also an EDA finding: `near_dup_pairs.csv`
gives you source/rewrite pairs to diff when answering *"what makes fusion different?"*.

In [6]:
drop_ids = set(pairs["train_id"]) if len(pairs) else set()
train_clean = train[~train["id"].isin(drop_ids)]
print(f"dropped {len(drop_ids):,} training rows -> train={len(train_clean):,}, val={len(val):,}")

share = pd.concat({"train": train_clean.label.value_counts(normalize=True),
                   "val": val.label.value_counts(normalize=True)}, axis=1).loc[LABELS]
print(share.round(3).to_string())

dropped 1,263 training rows -> train=38,737, val=10,000
        train    val
label               
human   0.241  0.245
ai      0.460  0.448
fusion  0.298  0.307


In [7]:
SPLITS_DIR.mkdir(exist_ok=True)
target = SPLITS_DIR / "val_ids.csv"
if target.exists() and not OVERWRITE:
    print("splits/ already exists - kept the existing fixed split (set OVERWRITE = True to replace it)")
else:
    train_clean[["id"]].to_csv(SPLITS_DIR / "train_ids.csv", index=False)
    val[["id"]].to_csv(SPLITS_DIR / "val_ids.csv", index=False)
    if len(pairs):
        pairs.drop(columns=["tr_pos", "va_pos"]).to_csv(SPLITS_DIR / "near_dup_pairs.csv", index=False)
    print("saved:", *sorted(p.name for p in SPLITS_DIR.iterdir()))

saved: near_dup_pairs.csv train_ids.csv val_ids.csv
